## tl;dr

V11.19.1 pairwise ranking / 成对排序。This verifies saved evidence,not a new experiment or Alpha certificate.

Observed complete exploratory survivors:0/2;certified usable Alpha:0.


## Context & Methods

24 continuousCNY3m accounts,two primary identities,2023–2024 reused development,2022 training.
### Key Assumptions

Overlapping pairs are not independent observations;gross proxy labels omit execution costs.
The initial numerical abort charged24Trials;corrected24adds to debt3648. No2025/26 access.
RequiresPython3.10+ andDuckDB; install project research extras if missing.


## Data

### 1. Check the immutable result and independent audit


In [1]:
import json,hashlib,math
from pathlib import Path
import duckdb
repo = Path.cwd()
if not (repo/'docs/V11_19_RESULT.summary.json').exists():
    repo = repo.parent
operation = repo/'artifacts/pairwise-ranking/epoch-002'
summary = json.loads((repo/'docs/V11_19_RESULT.summary.json').read_text(encoding='utf-8'))
audit = json.loads((operation/'INDEPENDENT_AUDIT.json').read_text(encoding='utf-8'))
assert audit['pass'] and summary['independent_audit_pass']
assert hashlib.sha256((operation/'RESULT.json').read_bytes()).hexdigest()==summary['source_result_sha256']
assert len(summary['rows'])==24 and summary['raw_trial_lower_bound']==3648
print({'accounts':24,'debt':3648,'training_pairs':summary['training_pairs'],'rank_dates':summary['rank_dates']})


{'accounts': 24, 'debt': 3648, 'training_pairs': 56320, 'rank_dates': 189}


### 2. Check source-query identity and training horizons

This binds the completed raw-source audit without repeating its market query.


In [2]:
assert (repo/'scripts/pairwise_source_audit.sql').read_text(encoding='utf-8')==audit['source_query']
for key,m in summary['models_summary'].items():
    assert m['maximum_label_end']<=m['fit_cutoff']<str(m['year'])+'-01-01'
    assert m['training_signal_dates']>=30
print({'models':len(summary['models_summary']),'mature_prefixes_verified':True})


{'models': 16, 'mature_prefixes_verified': True}


## Results

### 3. Reconcile all24 saved accounts using independent SQL


In [3]:
query = (repo/'scripts/lead_challenge_audit.sql').read_text(encoding='utf-8')
query = query.replace('__ACCOUNT_GLOB__',(operation/'accounts/*.jsonl').as_posix())
with duckdb.connect() as con:
    cur=con.execute(query)
    fields=[d[0] for d in cur.description]
    actual={r[0]:dict(zip(fields,r)) for r in cur.fetchall()}
assert len(actual)==24
for row in summary['rows']:
    for key in ('net_return','final_nav','cost_cny','max_drawdown'):
        assert math.isclose(row[key],actual[row['account_key']][key],rel_tol=1e-10,abs_tol=1e-6)
print({'account_aggregate_checks':24,'pass':True})


{'account_aggregate_checks': 24, 'pass': True}


### 4. Inspect every primary result at both costs


In [4]:
for row in summary['primary_comparisons']:
    print({k:row[k] for k in ('identity','roundtrip_bps','return2023','return2024','net_return','sharpe','minimum_control_increment','failed_gates')})
print({'exploratory_survivors':summary['screen_survived'],'validated_alpha':summary['validated_alpha']})


{'identity': 'linear', 'roundtrip_bps': 164, 'return2023': -0.15413666257017977, 'return2024': -0.1605295599307529, 'net_return': -0.2899227318893476, 'sharpe': -0.4342638838208464, 'minimum_control_increment': -0.49925973216535047, 'failed_gates': 'annual_increment, both_years_positive, drawdown, sharpe, total_increment'}
{'identity': 'linear', 'roundtrip_bps': 82, 'return2023': -0.069007060390054, 'return2024': -0.07322115688339048, 'net_return': -0.137175440478563, 'sharpe': -0.10283683132990858, 'minimum_control_increment': -0.4577785574844231, 'failed_gates': 'annual_increment, both_years_positive, drawdown, sharpe, total_increment'}
{'identity': 'quadratic', 'roundtrip_bps': 164, 'return2023': -0.16100111915575732, 'return2024': -0.2001164894083618, 'net_return': -0.32889862980785234, 'sharpe': -0.5935993259980636, 'minimum_control_increment': -0.5382356300838552, 'failed_gates': 'annual_increment, both_years_positive, drawdown, sharpe, total_increment'}
{'identity': 'quadratic',

## Takeaways

Only full survivors can be frozen for preregistered deeper challenges;history reuse is not independentOOS.
Cells execute in order under ordinaryPython. nbformat,nbclient,ipykernel are absent on the checked host;Jupyter frontend/kernel QA is NOT_RUN.
After installing optional dependencies: `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/V11_19_AUDIT.ipynb`.
